In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Discrete norms
# ---------------------------------------------------------

def norm_h(u, h):
    weights = np.ones_like(u, dtype=float)
    weights[0] = 0.5
    weights[-1] = 0.5
    return np.sqrt(h * np.sum(weights * np.abs(u)**2))


def norm_4_h(u, h):
    weights = np.ones_like(u, dtype=float)
    weights[0] = 0.5
    weights[-1] = 0.5
    return (h * np.sum(weights * np.abs(u)**4))**0.25


def D_h_plus(u, h):
    return (u[1:] - u[:-1]) / h


def norm_h_plus(p, h):
    return np.sqrt(h * np.sum(np.abs(p)**2))


def gn_quotient(u, h):
    lhs = norm_4_h(u, h)**4
    rhs = norm_h(u, h)**3 * norm_h_plus(D_h_plus(u, h), h) + norm_h(u, h)**4
    return lhs / rhs


# ---------------------------------------------------------
# Random tests for several mesh sizes
# ---------------------------------------------------------

def test_gn_for_meshes(J_values, L=10.0, ntests=1000, complex_vectors=True, seed=67):
    rng = np.random.default_rng(seed)

    max_ratios = []
    mean_ratios = []
    p95_ratios = []

    for J in J_values:
        h = 2 * L / J
        ratios = []

        for _ in range(ntests):
            if complex_vectors:
                u = rng.normal(size=J+1) + 1j * rng.normal(size=J+1)
            else:
                u = rng.normal(size=J+1)

            ratios.append(gn_quotient(u, h))

        ratios = np.array(ratios)

        max_ratios.append(ratios.max())
        mean_ratios.append(ratios.mean())
        p95_ratios.append(np.quantile(ratios, 0.95))

        print(
            f"J={J:5d}, h={h:.3e}, "
            f"max Q={ratios.max():.6e}, "
            f"mean Q={ratios.mean():.6e}, "
            f"95% Q={np.quantile(ratios, 0.95):.6e}"
        )

    return np.array(max_ratios), np.array(mean_ratios), np.array(p95_ratios)


J_values = [50, 100, 200, 400, 800, 1600]
L = 10.0
ntests = 10_000

max_Q, mean_Q, p95_Q = test_gn_for_meshes(
    J_values=J_values,
    L=L,
    ntests=ntests,
    complex_vectors=True,
    seed=67,
)

h_values = np.array([2 * L / J for J in J_values])


# ---------------------------------------------------------
# Plot: quotient versus h
# ---------------------------------------------------------

plt.figure()#figsize=(6, 4))
plt.loglog(h_values, max_Q, "o-", label="max quotient")
plt.loglog(h_values, p95_Q, "s--", label="95% quotient")
plt.loglog(h_values, mean_Q, "d--", label="mean quotient")

plt.gca().invert_xaxis()

plt.xticks(h_values, [f"{h:.3g}" for h in h_values])
plt.xlabel(r"$h$  (decreases to the right)")
plt.ylabel(r"$\frac{\|U\|_{4,h}^{4}}{\|U\|_{h}^{3}\|D_h^+U\|_{h,+}+\|U\|_{h}^{4}}$", rotation=0, labelpad=20)
plt.title("Discrete Gagliardo-Nirenberg quotient vs mesh size")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Optional: global empirical bound across all tested meshes
# ---------------------------------------------------------

C_num = 1.05 * np.max(max_Q)

print("\nEmpirical bound across all tested meshes:")
print("C_num =", C_num)
print("All tested mesh maxima <= C_num:", np.all(max_Q <= C_num))

In [ ]:
# ---------------------------------------------------------
# Plot LHS vs RHS for the discrete Gagliardo-Nirenberg inequality
# ---------------------------------------------------------

def gn_lhs_rhs(u, h):
    lhs = norm_4_h(u, h)**4
    rhs = norm_h(u, h)**3 * norm_h_plus(D_h_plus(u, h), h) + norm_h(u, h)**4
    return lhs, rhs


def collect_lhs_rhs(J_values, L=10.0, ntests=500, complex_vectors=True, seed=67):
    rng = np.random.default_rng(seed)

    all_lhs = []
    all_rhs = []
    all_h = []

    for J in J_values:
        h = 2 * L / J

        for _ in range(ntests):
            if complex_vectors:
                u = rng.normal(size=J+1) + 1j * rng.normal(size=J+1)
            else:
                u = rng.normal(size=J+1)

            lhs, rhs = gn_lhs_rhs(u, h)

            all_lhs.append(lhs)
            all_rhs.append(rhs)
            all_h.append(h)

    return np.array(all_lhs), np.array(all_rhs), np.array(all_h)


lhs_vals, rhs_vals, h_tags = collect_lhs_rhs(
    J_values=J_values,
    L=L,
    ntests=10_000,
    complex_vectors=True,
    seed=67,
)

C_values = [.05, .1]

plt.figure(figsize=(6, 5))

for h in np.unique(h_tags):
    mask = np.isclose(h_tags, h)
    plt.loglog(
        rhs_vals[mask],
        lhs_vals[mask],
        ".",
        alpha=0.35,
        label=fr"$h={h:.3g}$",
    )

xline = np.logspace(
    np.log10(rhs_vals.min()),
    np.log10(rhs_vals.max()),
    300,
)

for C in C_values:
    plt.loglog(
        xline,
        C * xline,
        "--",
        linewidth=2,
        label=rf"$y={C}x$",
    )

plt.xlabel(
    r"$\|U\|_{h}^{3}\|D_h^+U\|_{h,+}+\|U\|_{h}^{4}$"
)

plt.ylabel(
    r"$\|U\|_{4,h}^{4}$",
    rotation=0,
    labelpad=45,
)

plt.title("Discrete Gagliardo-Nirenberg inequality: LHS vs RHS")
plt.grid(True, which="both", alpha=0.3)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


ratios = lhs_vals / rhs_vals

print("Max LHS/RHS over all tests:", ratios.max())
print("Mean LHS/RHS over all tests:", ratios.mean())

for C in C_values:
    print(f"Percentage satisfying LHS <= {C} RHS:", 100 * np.mean(ratios <= C), "%")